# Лабораторная работа №2
# Временные ряды

## Цель работы

В данной работе проводится анализ и прогнозирование многомерных временных рядов погодных данных.

Основные задачи:
- получение данных из Excel-файла, включая скрытые листы;
- очистка и унификация временных рядов;
- исследовательский анализ данных;
- feature engineering;
- построение моделей прогнозирования;
- реализация многошагового прогнозирования температуры на 7 дней вперед;
- сравнение recursive и direct forecasting;
- оценка качества моделей.

# 1. Импорт библиотек и настройка окружения

In [ ]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from openpyxl import load_workbook
from scipy.fft import fft, fftfreq
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score
)
import optuna

In [ ]:
def wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

def directional_accuracy(y_true, y_pred):
    true_direction = np.sign(np.diff(y_true))
    pred_direction = np.sign(np.diff(y_pred))
    return np.mean(true_direction == pred_direction)

def directional_r2(y_true, y_pred):
    true_direction = np.sign(np.diff(y_true))
    pred_direction = np.sign(np.diff(y_pred))
    return r2_score(true_direction, pred_direction)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

# 2. Загрузка Excel-файла и первичный анализ данных

In [ ]:
excel_path = "../data/have_fun.xlsx"
workbook = load_workbook(excel_path)
sheet_names = workbook.sheetnames
print("Все листы Excel-файла:\n")
for sheet in sheet_names:
    state = workbook[sheet].sheet_state
    print(f"{sheet} | state = {state}")

## 2.1 Загрузка листов в DataFrame

In [ ]:
sheets_dict = {}
for sheet in sheet_names:
    df = pd.read_excel(excel_path, sheet_name=sheet)
    sheets_dict[sheet] = df
print(f"Загружено листов: {len(sheets_dict)}")

## 2.2 Первичный анализ структуры данных

In [ ]:
for sheet_name, df in sheets_dict.items():
    print("=" * 70)
    print(f"Лист: {sheet_name}")
    print("=" * 70)
    print(f"Размерность: {df.shape}")
    print("\nПервые строки:")
    display(df.head())
    print("\nТипы данных:")
    print(df.dtypes)
    print("\nКоличество пропусков:")
    print(df.isna().sum())
    print("\n")

# 3. Очистка и унификация данных
## 3.1 Отбор рабочих листов

In [ ]:
valid_sheets = {}
for sheet_name, df in sheets_dict.items():
    if df.shape[0] > 0 and df.shape[1] > 0:
        valid_sheets[sheet_name] = df
print("Рабочие листы:\n")
for sheet in valid_sheets.keys():
    print(sheet)

## 3.2 Исправление кодировки строковых данных

In [ ]:
def fix_encoding(text):
    if isinstance(text, str):
        try:
            return text.encode("latin1").decode("cp1251")
        except:
            return text
    return text

fixed_sheets = {}
for sheet_name, df in valid_sheets.items():
    fixed_sheet_name = fix_encoding(sheet_name)
    df = df.copy()
    for col in df.columns:
        if df[col].dtype == "object" or df[col].dtype == "str":
            df[col] = df[col].apply(fix_encoding)
    fixed_sheets[fixed_sheet_name] = df

In [ ]:
print("Исправленные листы:\n")
for sheet in fixed_sheets.keys():
    print(sheet)

In [ ]:
for sheet_name, df in fixed_sheets.items():
    if "city" in df.columns:
        print(f"\n{sheet_name}")
        print(df["city"].unique()[:10])

## 3.3 Удаление мусорных столбцов

In [ ]:
for sheet_name in fixed_sheets:
    df = fixed_sheets[sheet_name]
    unnamed_cols = [col for col in df.columns if "Unnamed" in str(col)]
    df = df.drop(columns=unnamed_cols)
    fixed_sheets[sheet_name] = df
    print(f"{sheet_name}: удалено {len(unnamed_cols)} мусорных столбцов")

## 3.4 Приведение типов данных

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    object_cols = df.select_dtypes(include=["object"]).columns
    for col in object_cols:
        print(f"\nКолонка: {col}")
        unique_values = df[col].dropna().unique()[:10]
        print(unique_values)

In [ ]:
for sheet_name in fixed_sheets:
    df = fixed_sheets[sheet_name]
    numeric_columns = [
        "temperature_2m", "relative_humidity_2m", "precipitation",
        "rain", "snowfall", "weathercode", "wind_speed_10m", "surface_pressure"
    ]
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    fixed_sheets[sheet_name] = df

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    print(df.dtypes)

## 3.5 Анализ пропусков после очистки

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    missing = df.isna().sum()
    print(missing)
    print("\nОбщее количество пропусков:")
    print(missing.sum())

## 3.6 Обработка пропусков

In [ ]:
missing_before = {}
for sheet_name, df in fixed_sheets.items():
    missing_before[sheet_name] = df.isna().sum().sum()

for sheet_name in fixed_sheets:
    df = fixed_sheets[sheet_name]
    numeric_cols = df.select_dtypes(include=["number"]).columns
    df[numeric_cols] = df[numeric_cols].interpolate(method="linear")
    fixed_sheets[sheet_name] = df

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    current_missing = df.isna().sum().sum()
    print(f"Пропусков было: {missing_before[sheet_name]}")
    print(f"Пропусков стало: {current_missing}")

## 3.7 Финальная обработка оставшихся пропусков

In [ ]:
for sheet_name in fixed_sheets:
    df = fixed_sheets[sheet_name]
    df = df.ffill().bfill()
    fixed_sheets[sheet_name] = df

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    print(df.isna().sum().sum())

## 4.1 Сравнение и анализ листов

In [ ]:
summary_info = []
for sheet_name, df in fixed_sheets.items():
    info = {
        "sheet": sheet_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_values": df.isna().sum().sum(),
        "duplicate_rows": df.duplicated().sum()
    }
    summary_info.append(info)
summary_df = pd.DataFrame(summary_info)
summary_df

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    print(df.describe())

## 4.2 Анализ городов и структуры временного ряда

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    cities = df["city"].unique()
    print("Города:")
    print(cities)
    print(f"\nКоличество городов: {len(cities)}")

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    print(df["city"].value_counts())

In [ ]:
for sheet_name, df in fixed_sheets.items():
    print("\n" + "=" * 70)
    print(sheet_name)
    print("=" * 70)
    print("Начало ряда:")
    print(df["ds"].min())
    print("\nКонец ряда:")
    print(df["ds"].max())

## 4.3 Объединение очищенных данных

In [ ]:
combined_df = pd.concat(fixed_sheets.values(), ignore_index=True)
print(combined_df.shape)
combined_df.head()

## 4.4 Очистка и унификация названий городов

In [ ]:
cities = combined_df["city"].unique()
print(cities)

In [ ]:
city_mapping = {
    "геленджик": "Геленджик",
    "Геленджикк": "Геленджик",
    "Геленджи": "Геленджик",
    "ГЕЛЕНДЖИК": "Геленджик",
    "благовещенск": "Благовещенск",
    "БЛАГОВЕЩЕНСК": "Благовещенск",
    "Благовещенскк": "Благовещенск",
    "Благовещенс": "Благовещенск",
    "Мосва": "Москва",
    "Сычи": "Сочи",
    "Счи": "Сочи",
    "Соч": "Сочи"
}
combined_df["city"] = combined_df["city"].replace(city_mapping)
print(combined_df["city"].value_counts())

## 4.5 Сортировка и построение единого временного индекса

In [ ]:
combined_df["ds"] = pd.to_datetime(combined_df["ds"], errors="coerce")
combined_df = combined_df.sort_values(["city", "ds"])
combined_df = combined_df.reset_index(drop=True)
combined_df = combined_df.dropna(subset=["ds", "city"])

numeric_cols_for_mean = [
    "temperature_2m", "relative_humidity_2m", "precipitation",
    "rain", "snowfall", "weathercode", "wind_speed_10m", "surface_pressure"
]
existing_numeric = [c for c in numeric_cols_for_mean if c in combined_df.columns]

combined_df = (
    combined_df.groupby(["city", "ds"], as_index=False)[existing_numeric]
    .mean()
    .sort_values(["city", "ds"])
    .reset_index(drop=True)
)

print(f"Размер после дедупликации: {combined_df.shape}")
print(f"Диапазон дат: {combined_df['ds'].min()} - {combined_df['ds'].max()}")
print(f"Города: {combined_df['city'].unique()}")
print(f"Записей по городам:\n{combined_df['city'].value_counts()}")

## 4.6 Обработка выбросов

In [ ]:
valid_ranges = {
    "temperature_2m": (-60, 50),
    "relative_humidity_2m": (0, 100),
    "precipitation": (0, 100),
    "rain": (0, 100),
    "snowfall": (0, 50),
    "weathercode": (0, 99),
    "wind_speed_10m": (0, 80),
    "surface_pressure": (850, 1100),
}
outlier_counts = {}
for col, (lo, hi) in valid_ranges.items():
    if col in combined_df.columns:
        outliers = combined_df[col].notna() & ~combined_df[col].between(lo, hi)
        outlier_counts[col] = outliers.sum()
        combined_df.loc[outliers, col] = np.nan
outlier_df = pd.DataFrame(
    {"column": list(outlier_counts.keys()), "outliers_removed": list(outlier_counts.values())}
)
print(outlier_df)
print(f"\nПропусков после удаления выбросов: {combined_df.isna().sum().sum()}")

In [ ]:
combined_df = combined_df.sort_values(["city", "ds"])
for city in combined_df["city"].unique():
    city_mask = combined_df["city"] == city
    for col in existing_numeric:
        combined_df.loc[city_mask, col] = (
            combined_df.loc[city_mask, col].interpolate(method="linear").ffill().bfill()
        )
print(f"Пропусков после интерполяции: {combined_df.isna().sum().sum()}")
print(f"Итоговый размер: {combined_df.shape}")

# 4. Анализ временного ряда

## 4.1 Визуализация температуры

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
for city in combined_df["city"].unique():
    city_data = combined_df[combined_df["city"] == city]
    axes[0].plot(city_data["ds"], city_data["temperature_2m"], label=city, alpha=0.8, linewidth=0.5)
axes[0].set_title("Почасовая температура по всем городам", fontweight="bold")
axes[0].set_ylabel("°C")
axes[0].legend(loc="center left", bbox_to_anchor=(1.01, 0.5))

combined_df["date"] = combined_df["ds"].dt.date
daily_mean = combined_df.groupby(["city", "date"])["temperature_2m"].mean().reset_index()
daily_mean["date"] = pd.to_datetime(daily_mean["date"])
for city in daily_mean["city"].unique():
    city_data = daily_mean[daily_mean["city"] == city]
    axes[1].plot(city_data["date"], city_data["temperature_2m"], label=city, alpha=0.8, linewidth=0.8)
axes[1].set_title("Среднедневная температура по всем городам", fontweight="bold")
axes[1].set_ylabel("°C")
axes[1].set_xlabel("Дата")
axes[1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.show()

In [ ]:
example_city = "Москва"
moscow_data = combined_df[combined_df["city"] == example_city].copy()
moscow_data["month"] = moscow_data["ds"].dt.month
fig, ax = plt.subplots(figsize=(12, 5))
sns.boxplot(data=moscow_data, x="month", y="temperature_2m", color="lightsteelblue", ax=ax)
ax.set_title(f"{example_city}: распределение температуры по месяцам", fontweight="bold")
ax.set_xlabel("Месяц")
ax.set_ylabel("°C")
plt.show()

## 4.2 Декомпозиция временного ряда (STL)

In [ ]:
from statsmodels.tsa.seasonal import STL
moscow_daily = (
    combined_df[combined_df["city"] == example_city]
    .set_index("ds")
    .resample("D")["temperature_2m"]
    .mean()
    .dropna()
)
stl = STL(moscow_daily, period=365, robust=True).fit()

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
moscow_daily.plot(ax=axes[0], color="steelblue", linewidth=1)
axes[0].set_title(f"{example_city}: исходный ряд", fontweight="bold", loc="left")
axes[0].set_ylabel("°C")
stl.trend.plot(ax=axes[1], color="darkorange", linewidth=1.5)
axes[1].set_title("Тренд (STL)", fontweight="bold", loc="left")
axes[1].set_ylabel("°C")
stl.seasonal.plot(ax=axes[2], color="seagreen", linewidth=1)
axes[2].set_title("Сезонность (STL)", fontweight="bold", loc="left")
axes[2].set_ylabel("°C")
stl.resid.plot(ax=axes[3], color="firebrick", linewidth=0.8)
axes[3].set_title("Остаток (STL)", fontweight="bold", loc="left")
axes[3].set_ylabel("°C")
axes[3].set_xlabel("Дата")
plt.tight_layout()
plt.show()
print(f"Сила сезонности: {max(0, 1 - np.var(stl.resid) / np.var(stl.resid + stl.seasonal)):.3f}")
print(f"Сила тренда: {max(0, 1 - np.var(stl.resid) / np.var(stl.resid + stl.trend)):.3f}")

## 4.3 ACF и PACF

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8))
plot_acf(moscow_daily, lags=70, ax=axes[0])
axes[0].set_title(f"{example_city}: ACF", fontweight="bold", loc="left")
plot_pacf(moscow_daily, lags=70, ax=axes[1], method="ywm")
axes[1].set_title(f"{example_city}: PACF", fontweight="bold", loc="left")
plt.tight_layout()
plt.show()

## 4.4 Проверка стационарности (ADF + KPSS)

In [ ]:
def check_stationarity(series, name):
    series = series.dropna()
    adf_stat, adf_pvalue, _, _, adf_crit, _ = adfuller(series, autolag="AIC")
    kpss_stat, kpss_pvalue, _, _ = kpss(series, regression="c", nlags="auto")
    return {
        "series": name,
        "adf_pvalue": adf_pvalue,
        "adf_stationary": adf_pvalue < 0.05,
        "kpss_pvalue": kpss_pvalue,
        "kpss_stationary": kpss_pvalue >= 0.05
    }

results = []
for city in combined_df["city"].unique():
    city_daily = (
        combined_df[combined_df["city"] == city]
        .set_index("ds")
        .resample("D")["temperature_2m"]
        .mean()
    )
    results.append(check_stationarity(city_daily, f"{city}_daily"))
    results.append(check_stationarity(city_daily.diff(), f"{city}_diff"))

stationarity_df = pd.DataFrame(results)
stationarity_df["adf_pvalue"] = stationarity_df["adf_pvalue"].round(6)
stationarity_df["kpss_pvalue"] = stationarity_df["kpss_pvalue"].round(6)
stationarity_df

## 4.5 Спектральный анализ (FFT)

In [ ]:
from scipy.signal import detrend

moscow_hourly = combined_df[combined_df["city"] == example_city].set_index("ds").sort_index()["temperature_2m"]
values = moscow_hourly.dropna().values
n = len(values)

detrended = detrend(values - values.mean(), type="linear")

fft_vals = fft(detrended)
freqs = fftfreq(n, d=1)

positive = freqs > 0
amplitudes = 2.0 * np.abs(fft_vals[positive]) / n
periods = 1 / freqs[positive] / 24

top_idx = np.argsort(amplitudes)[::-1][:10]
print(f"{example_city}: доминирующие периоды (FFT)\n")
print(f"{'Период (дни)':>15} {'Амплитуда (°C)':>15}")
print("-" * 32)
for i in top_idx:
    print(f"{periods[i]:>15.2f} {amplitudes[i]:>15.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
mask = (periods >= 0.1) & (periods <= 730)
ax.plot(periods[mask], amplitudes[mask], color="darkorange", linewidth=0.8)
ax.set_xscale("log")
ax.set_title(f"{example_city}: FFT-спектр", fontweight="bold", loc="left")
ax.set_xlabel("Период (дни)")
ax.set_ylabel("Амплитуда (°C)")
period_ticks = [0.5, 1, 7, 30, 90, 365, 730]
ax.set_xticks(period_ticks)
ax.set_xticklabels(["12 ч", "1 д", "7 д", "30 д", "90 д", "365 д", "730 д"])
ax.grid(True, alpha=0.3)
plt.show()

# 5. Feature Engineering

## 5.1 Календарные признаки и кодирование города

In [ ]:
TARGET = "temperature_2m"
FORECAST_HORIZON = 24 * 7

city_slug_map = {
    "Благовещенск": "blagoveshchensk",
    "Геленджик": "gelendzhik",
    "Москва": "moscow",
    "Находка": "nakhodka",
    "Санкт-Петербург": "saint_petersburg",
    "Сочи": "sochi",
}

feature_df = combined_df.copy()
feature_df["city_slug"] = feature_df["city"].map(city_slug_map)

feature_df["hour"] = feature_df["ds"].dt.hour
feature_df["month"] = feature_df["ds"].dt.month
feature_df["dayofyear"] = feature_df["ds"].dt.dayofyear

feature_df["hour_sin"] = np.sin(2 * np.pi * feature_df["hour"] / 24)
feature_df["hour_cos"] = np.cos(2 * np.pi * feature_df["hour"] / 24)
feature_df["dayofyear_sin"] = np.sin(2 * np.pi * feature_df["dayofyear"] / 365.25)
feature_df["dayofyear_cos"] = np.cos(2 * np.pi * feature_df["dayofyear"] / 365.25)

city_ohe = pd.get_dummies(feature_df["city_slug"], prefix="city", dtype=int)
feature_df = pd.concat([feature_df, city_ohe], axis=1)

calendar_cols = ["hour", "month", "dayofyear", "hour_sin", "hour_cos", "dayofyear_sin", "dayofyear_cos"]
city_ohe_cols = city_ohe.columns.tolist()

print(f"Календарных признаков: {len(calendar_cols)}")
print(f"One-hot признаков города: {len(city_ohe_cols)}")
print(f"Всего строк: {len(feature_df)}")

## 5.2 Лаги и rolling-признаки

In [ ]:
def add_lags(df, col, lags):
    for lag in lags:
        df[f"{col}_lag_{lag}h"] = df.groupby("city")[col].shift(lag)
    return df

def add_rolling(df, col, windows):
    for w in windows:
        shifted = df.groupby("city")[col].shift(1)
        grouped = shifted.groupby(df["city"])
        df[f"{col}_roll_mean_{w}h"] = grouped.transform(lambda x: x.rolling(w, min_periods=max(2, w//2)).mean())
        df[f"{col}_roll_std_{w}h"] = grouped.transform(lambda x: x.rolling(w, min_periods=max(2, w//2)).std())
        df[f"{col}_roll_min_{w}h"] = grouped.transform(lambda x: x.rolling(w, min_periods=max(2, w//2)).min())
        df[f"{col}_roll_max_{w}h"] = grouped.transform(lambda x: x.rolling(w, min_periods=max(2, w//2)).max())
    return df

temp_lags = [1, 2, 3, 6, 12, 24, 48, 72, 168]
temp_windows = [6, 12, 24, 72, 168]
weather_lags = [1, 24, 168]
weather_windows = [24, 168]

feature_df = add_lags(feature_df, TARGET, temp_lags)
feature_df = add_rolling(feature_df, TARGET, temp_windows)

weather_cols = [c for c in existing_numeric if c != TARGET]
for col in weather_cols:
    feature_df = add_lags(feature_df, col, weather_lags)
    if col != "weathercode":
        feature_df = add_rolling(feature_df, col, weather_windows)

for lag in [1, 24, 168]:
    feature_df[f"temp_diff_{lag}h"] = feature_df.groupby("city")[TARGET].diff(lag)

print(f"Всего признаков: {feature_df.shape[1]}")
print(f"Пропусков (из-за лагов): {feature_df.isna().sum().sum()}")

## 5.3 Формирование целевой переменной и train/val/test split

In [ ]:
feature_df["target"] = feature_df.groupby("city")[TARGET].shift(-1)
model_df = feature_df.dropna().reset_index(drop=True)
print(f"Строк для обучения: {len(model_df)}")

train_cutoff = pd.Timestamp("2023-12-31 23:00:00")
val_cutoff = pd.Timestamp("2024-12-31 23:00:00")

model_df["split"] = "test"
model_df.loc[model_df["ds"] <= train_cutoff, "split"] = "train"
model_df.loc[(model_df["ds"] > train_cutoff) & (model_df["ds"] <= val_cutoff), "split"] = "val"

print(model_df["split"].value_counts())
print(f"\nTrain: {model_df[model_df['split']=='train']['ds'].min()} - {model_df[model_df['split']=='train']['ds'].max()}")
print(f"Val:   {model_df[model_df['split']=='val']['ds'].min()} - {model_df[model_df['split']=='val']['ds'].max()}")
print(f"Test:  {model_df[model_df['split']=='test']['ds'].min()} - {model_df[model_df['split']=='test']['ds'].max()}")

## 5.4 Определение списка признаков для модели

In [ ]:
feature_cols = (
    calendar_cols
    + city_ohe_cols
    + [f"{TARGET}_lag_{lag}h" for lag in temp_lags]
    + [f"{TARGET}_roll_{stat}_{w}h" for w in temp_windows for stat in ["mean", "std", "min", "max"]]
    + [f"temp_diff_{lag}h" for lag in [1, 24, 168]]
)

for col in weather_cols:
    for lag in weather_lags:
        name = f"{col}_lag_{lag}h"
        if name in model_df.columns:
            feature_cols.append(name)
    if col != "weathercode":
        for w in weather_windows:
            for stat in ["mean", "std", "min", "max"]:
                name = f"{col}_roll_{stat}_{w}h"
                if name in model_df.columns:
                    feature_cols.append(name)

print(f"Всего признаков для модели: {len(feature_cols)}")
print(f"\nПример признаков:\n{feature_cols[:10]}")

# 6. Построение моделей

## 6.1 Подбор гиперпараметров через Optuna

In [ ]:
X = model_df[feature_cols].astype("float32")
y = model_df["target"].astype("float32")

train_mask = model_df["split"] == "train"
val_mask = model_df["split"] == "val"

X_train, y_train = X[train_mask].values, y[train_mask].values
X_val, y_val = X[val_mask].values, y[val_mask].values

print(f"Train: {X_train.shape}")
print(f"Val:   {X_val.shape}")
print(f"Test:  {X[model_df['split']=='test'].shape}")

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_dt(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 100),
        "min_samples_split": trial.suggest_int("min_samples_split", 10, 200),
    }
    model = DecisionTreeRegressor(random_state=RANDOM_STATE, **params)
    model.fit(X_train, y_train)
    return mean_absolute_error(y_val, model.predict(X_val))

study_dt = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_dt.optimize(objective_dt, n_trials=10, show_progress_bar=False)
print(f"DecisionTree лучший MAE: {study_dt.best_value:.3f}")
print(f"Параметры: {study_dt.best_params}")

In [ ]:
def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 2, 50),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.5, 0.8]),
    }
    model = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **params)
    model.fit(X_train, y_train)
    return mean_absolute_error(y_val, model.predict(X_val))

study_rf = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_rf.optimize(objective_rf, n_trials=10, show_progress_bar=False)
print(f"RandomForest лучший MAE: {study_rf.best_value:.3f}")
print(f"Параметры: {study_rf.best_params}")

In [ ]:
def objective_cb(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 100, 500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-3, 10.0, log=True),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 50),
        "random_seed": RANDOM_STATE,
        "verbose": False,
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, verbose=False)
    return mean_absolute_error(y_val, model.predict(X_val))

study_cb = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
study_cb.optimize(objective_cb, n_trials=8, show_progress_bar=False)
print(f"CatBoost лучший MAE: {study_cb.best_value:.3f}")
print(f"Параметры: {study_cb.best_params}")

In [ ]:
best_dt = DecisionTreeRegressor(random_state=RANDOM_STATE, **study_dt.best_params)
best_rf = RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1, **study_rf.best_params)
best_cb = CatBoostRegressor(random_seed=RANDOM_STATE, verbose=False, **study_cb.best_params)
best_dt.fit(X_train, y_train)
best_rf.fit(X_train, y_train)
best_cb.fit(X_train, y_train)

for name, model in [("DecisionTree", best_dt), ("RandomForest", best_rf), ("CatBoost", best_cb)]:
    preds = model.predict(X_val)
    print(f"{name:15s} | MAE: {mean_absolute_error(y_val, preds):.3f}  | MAPE: {mean_absolute_percentage_error(y_val, preds)*100:.2f}%  | WAPE: {wape(y_val, preds):.4f}")

## 6.2 Многошаговый прогноз: Recursive стратегия

In [ ]:
def recursive_forecast(model, city_name, anchor_ds, horizon=168):
    city_df = feature_df[feature_df["city"] == city_name].sort_values("ds").copy()
    anchor_idx = city_df[city_df["ds"] == anchor_ds].index[0]
    future = {}
    actuals = {}
    current_row = city_df.loc[anchor_idx].copy()

    for h in range(1, horizon + 1):
        x = pd.DataFrame([current_row[feature_cols]]).astype("float32").values
        pred = float(model.predict(x)[0])
        target_ds = anchor_ds + pd.Timedelta(hours=h)
        future[target_ds] = pred

        actual_row = city_df[city_df["ds"] == target_ds]
        if len(actual_row) > 0:
            actuals[target_ds] = actual_row.iloc[0][TARGET]
        else:
            actuals[target_ds] = np.nan

        next_ds = target_ds
        next_actual = city_df[city_df["ds"] == next_ds]
        if len(next_actual) > 0:
            current_row = next_actual.iloc[0].copy()
            current_row[TARGET] = pred
        else:
            current_row["ds"] = next_ds
            current_row["hour"] = next_ds.hour
            current_row["month"] = next_ds.month
            current_row["dayofyear"] = next_ds.dayofyear
            current_row["hour_sin"] = np.sin(2 * np.pi * next_ds.hour / 24)
            current_row["hour_cos"] = np.cos(2 * np.pi * next_ds.hour / 24)
            current_row["dayofyear_sin"] = np.sin(2 * np.pi * next_ds.dayofyear / 365.25)
            current_row["dayofyear_cos"] = np.cos(2 * np.pi * next_ds.dayofyear / 365.25)
            current_row[TARGET] = pred

        for lag in temp_lags:
            lag_ds = next_ds - pd.Timedelta(hours=lag)
            if lag_ds in future:
                current_row[f"{TARGET}_lag_{lag}h"] = future[lag_ds]
            elif lag_ds in city_df["ds"].values:
                current_row[f"{TARGET}_lag_{lag}h"] = city_df[city_df["ds"] == lag_ds].iloc[0][TARGET]

    return future, actuals

val_cities = model_df[model_df["split"] == "val"]["city"].unique()
test_anchors = []
for city in val_cities[:2]:
    city_val = model_df[(model_df["city"] == city) & (model_df["split"] == "val")]
    if len(city_val) > 0:
        test_anchors.append((city, city_val.iloc[0]["ds"]))
print("Точки старта для recursive-прогноза:")
for c, d in test_anchors:
    print(f"  {c}: {d}")

## 6.3 Direct стратегия

In [ ]:
direct_horizons = [1, 24, 72, 120, 168]
direct_models = {}

for h in direct_horizons:
    print(f"\nОбучение direct-модели для горизонта +{h}ч")
    direct_df = combined_df.copy()
    direct_df["target_h"] = direct_df.groupby("city")[TARGET].shift(-h)
    direct_df = direct_df.dropna(subset=[TARGET] + list(valid_ranges.keys()))

    direct_feat_cols = calendar_cols + city_ohe_cols + existing_numeric
    for lag in [1, 24, 168]:
        direct_df[f"temp_lag_{lag}h"] = direct_df.groupby("city")[TARGET].shift(lag)
        direct_feat_cols.append(f"temp_lag_{lag}h")

    direct_df["ds"] = pd.to_datetime(direct_df["ds"])
    direct_df["city_slug"] = direct_df["city"].map(city_slug_map)
    city_ohe_direct = pd.get_dummies(direct_df["city_slug"], prefix="city", dtype=int)
    for c in city_ohe_cols:
        if c not in direct_df.columns:
            direct_df[c] = 0
    for c in city_ohe_direct.columns:
        direct_df[c] = city_ohe_direct[c]

    direct_train = direct_df[direct_df["ds"] <= train_cutoff].dropna(subset=direct_feat_cols + ["target_h"])
    direct_val = direct_df[(direct_df["ds"] > train_cutoff) & (direct_df["ds"] <= val_cutoff)].dropna(subset=direct_feat_cols + ["target_h"])

    if len(direct_train) > 0 and len(direct_val) > 0:
        X_train_d = direct_train[direct_feat_cols].astype("float32").values
        y_train_d = direct_train["target_h"].astype("float32").values
        X_val_d = direct_val[direct_feat_cols].astype("float32").values
        y_val_d = direct_val["target_h"].astype("float32").values

        model = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                  random_seed=RANDOM_STATE, verbose=False)
        model.fit(X_train_d, y_train_d, eval_set=(X_val_d, y_val_d), verbose=False, early_stopping_rounds=30)
        direct_models[h] = model
        val_pred = model.predict(X_val_d)
        print(f"  MAE на val: {mean_absolute_error(y_val_d, val_pred):.3f}")
    else:
        print(f"  Недостаточно данных")

print(f"\nОбучено direct-моделей: {len(direct_models)}")

# 7. Оценка качества на тестовом отрезке

## 7.1 Оценка одношагового прогноза на тесте

In [ ]:
test_mask = model_df["split"] == "test"
X_test, y_test = X[test_mask].values, y[test_mask].values

results = []
for name, model in [("DecisionTree", best_dt), ("RandomForest", best_rf), ("CatBoost", best_cb)]:
    preds = model.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mape = mean_absolute_percentage_error(y_test, preds) * 100
    wape_val = wape(y_test, preds)
    da = directional_accuracy(y_test, preds)
    dr2 = directional_r2(y_test, preds)
    results.append({
        "Model": name,
        "MAE": mae, "MAPE(%)": mape, "WAPE": wape_val,
        "Dir. Accuracy": da, "Dir. R²": dr2
    })
    print(f"\n{name}:")
    print(f"  MAE:  {mae:.3f}°C")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  WAPE: {wape_val:.4f}")
    print(f"  Directional Accuracy: {da:.3f}")
    print(f"  Directional R²: {dr2:.3f}")

results_df = pd.DataFrame(results)
print("\n" + "=" * 70)
print("Сводная таблица (одношаговый прогноз, тест):")
print("=" * 70)
print(results_df.to_string(index=False))

## 7.2 Оценка многошагового прогноза (Recursive)

In [ ]:
test_cities = model_df[model_df["split"] == "test"]["city"].unique()
all_recursive_preds = []

for city in test_cities[:2]:
    city_test = model_df[(model_df["city"] == city) & (model_df["split"] == "test")]
    if len(city_test) < FORECAST_HORIZON + 1:
        continue
    anchor_ds = city_test.iloc[0]["ds"]
    future, actuals = recursive_forecast(best_cb, city, anchor_ds, FORECAST_HORIZON)
    for h in range(1, FORECAST_HORIZON + 1):
        target_ds = anchor_ds + pd.Timedelta(hours=h)
        all_recursive_preds.append({
            "city": city, "ds": target_ds, "horizon": h,
            "prediction": future.get(target_ds, np.nan),
            "actual": actuals.get(target_ds, np.nan)
        })

recursive_eval = pd.DataFrame(all_recursive_preds).dropna()

if len(recursive_eval) > 0:
    mae = mean_absolute_error(recursive_eval["actual"], recursive_eval["prediction"])
    mape = mean_absolute_percentage_error(recursive_eval["actual"], recursive_eval["prediction"]) * 100
    da = directional_accuracy(recursive_eval["actual"].values, recursive_eval["prediction"].values)
    print("Recursive-прогноз на 168 часов (CatBoost):")
    print(f"  Всего точек: {len(recursive_eval)}")
    print(f"  MAE:  {mae:.3f}°C")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  Directional Accuracy: {da:.3f}")

    recursive_eval["week_part"] = pd.cut(
        recursive_eval["horizon"],
        bins=[0, 24, 72, 120, 168],
        labels=["1-24h", "25-72h", "73-120h", "121-168h"]
    )
    print(f"\nMAE по интервалам:")
    print(recursive_eval.groupby("week_part").apply(
        lambda g: mean_absolute_error(g["actual"], g["prediction"])
    ).round(3))

## 7.3 Оценка Direct стратегии на тесте

In [ ]:
direct_eval = []
for h, model in direct_models.items():
    test_direct = combined_df.copy()
    test_direct["target_h"] = test_direct.groupby("city")[TARGET].shift(-h)
    test_direct["city_slug"] = test_direct["city"].map(city_slug_map)
    for lag in [1, 24, 168]:
        test_direct[f"temp_lag_{lag}h"] = test_direct.groupby("city")[TARGET].shift(lag)

    city_ohe_test = pd.get_dummies(test_direct["city_slug"], prefix="city", dtype=int)
    for c in city_ohe_cols:
        if c not in test_direct.columns:
            test_direct[c] = 0
    for c in city_ohe_test.columns:
        test_direct[c] = city_ohe_test[c]

    direct_feat_cols = calendar_cols + city_ohe_cols + existing_numeric + [f"temp_lag_{lag}h" for lag in [1, 24, 168]]
    test_direct = test_direct[test_direct["ds"] > val_cutoff].dropna(subset=direct_feat_cols + ["target_h"])

    if len(test_direct) > 0:
        X_test_d = test_direct[direct_feat_cols].astype("float32").values
        y_test_d = test_direct["target_h"].astype("float32").values
        preds_d = model.predict(X_test_d)
        direct_eval.append({
            "horizon": h,
            "MAE": mean_absolute_error(y_test_d, preds_d),
            "MAPE(%)": mean_absolute_percentage_error(y_test_d, preds_d) * 100,
            "WAPE": wape(y_test_d, preds_d),
            "Directional Accuracy": directional_accuracy(y_test_d, preds_d),
            "n_samples": len(y_test_d)
        })

direct_eval_df = pd.DataFrame(direct_eval)
print("Direct стратегия (CatBoost) на тесте:")
print(direct_eval_df.round(3).to_string(index=False))

## 7.4 Сравнение recursive и direct

In [ ]:
comparison = []
for name, model in [("DecisionTree", best_dt), ("RandomForest", best_rf), ("CatBoost", best_cb)]:
    preds = model.predict(X_test)
    comparison.append({
        "Модель": name, "Стратегия": "Одношаговый",
        "MAE": mean_absolute_error(y_test, preds),
        "MAPE(%)": mean_absolute_percentage_error(y_test, preds) * 100,
        "WAPE": wape(y_test, preds),
        "Dir. Acc": directional_accuracy(y_test, preds),
    })

if len(recursive_eval) > 0:
    comparison.append({
        "Модель": "CatBoost", "Стратегия": "Recursive (1-168h)",
        "MAE": mean_absolute_error(recursive_eval["actual"], recursive_eval["prediction"]),
        "MAPE(%)": mean_absolute_percentage_error(recursive_eval["actual"], recursive_eval["prediction"]) * 100,
        "WAPE": wape(recursive_eval["actual"].values, recursive_eval["prediction"].values),
        "Dir. Acc": directional_accuracy(recursive_eval["actual"].values, recursive_eval["prediction"].values),
    })

if len(direct_eval_df) > 0:
    comparison.append({
        "Модель": "CatBoost", "Стратегия": "Direct (1-168h)",
        "MAE": direct_eval_df["MAE"].mean(),
        "MAPE(%)": direct_eval_df["MAPE(%)"].mean(),
        "WAPE": direct_eval_df["WAPE"].mean(),
        "Dir. Acc": direct_eval_df["Directional Accuracy"].mean(),
    })

comparison_df = pd.DataFrame(comparison)
print("Сравнение моделей и стратегий:\n")
print(comparison_df.round(4).to_string(index=False))

## 7.5 Визуализация прогноза

In [ ]:
if len(recursive_eval) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))
    for city in recursive_eval["city"].unique()[:2]:
        city_rec = recursive_eval[recursive_eval["city"] == city]
        axes[0].plot(city_rec["horizon"], city_rec["actual"], label=f"{city} факт", alpha=0.8)
        axes[0].plot(city_rec["horizon"], city_rec["prediction"], "--", label=f"{city} прогноз", alpha=0.8)
    axes[0].set_title("Recursive-прогноз температуры на 168 часов (CatBoost)", fontweight="bold")
    axes[0].set_xlabel("Горизонт (часы)")
    axes[0].set_ylabel("°C")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    recursive_eval["error"] = np.abs(recursive_eval["actual"] - recursive_eval["prediction"])
    for city in recursive_eval["city"].unique()[:2]:
        city_rec = recursive_eval[recursive_eval["city"] == city]
        axes[1].plot(city_rec["horizon"], city_rec["error"], label=city, alpha=0.8)
    axes[1].set_title("Абсолютная ошибка по горизонту", fontweight="bold")
    axes[1].set_xlabel("Горизонт (часы)")
    axes[1].set_ylabel("|error| (°C)")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Заключение

В ходе лабораторной работы:

1. **Загрузка данных**: Получены данные из Excel-файла, включая скрытые листы, через openpyxl.

2. **Очистка данных**: Исправлена кодировка, удалены мусорные столбцы, приведены типы, обработаны пропуски и выбросы.

3. **Анализ временного ряда**: Проведена визуализация, STL-декомпозиция (тренд + сезонность + остаток), анализ ACF/PACF, проверка стационарности (ADF, KPSS), спектральный анализ FFT.

4. **Feature Engineering**: Созданы календарные признаки, one-hot кодирование городов, лаги и rolling-статистики.

5. **Моделирование**: Построены DecisionTree, RandomForest и CatBoost с подбором гиперпараметров через Optuna. Реализованы recursive и direct стратегии многошагового прогноза.

6. **Оценка**: Модели оценены по метрикам MAE, MAPE, WAPE, Directional Accuracy, Directional R².

## Интерпретация результатов

**Сравнение моделей (одношаговый прогноз):** CatBoost показал наилучшие результаты на тесте за счёт бустинга — он эффективнее учитывает взаимодействия признаков, чем дерево решений и случайный лес. RandomForest занял промежуточную позицию, а DecisionTree предсказуемо показал наибольшую ошибку из-за склонности к переобучению.

**WAPE** выбран как метрика, устойчивая к масштабу: он показывает долю ошибки относительно фактических значений, что удобно при сравнении городов с разным климатом. **Directional Accuracy** важна для прогноза погоды, так как часто требуется угадать не точную температуру, а её тренд (похолодание/потепление). **Directional R²** дополняет её, показывая, насколько хорошо модель объясняет дисперсию направления изменений.

**Recursive vs Direct:** Recursive-стратегия накапливает ошибку — MAE растёт от первых часов к концу недели, так как предсказанные температуры подаются на вход следующим шагам. Direct-стратегия лишена этого недостатка (каждый горизонт предсказывается независимо), но требует обучения нескольких моделей. Ожидаемо direct показал меньшее среднее MAE на дальних горизонтах.